# 01_data_ingestion.ipynb

This notebook downloads the German Credit dataset from OpenML, performs a basic
validation and uploads the raw dataset to Amazon S3.

## Import Dependencies

Import the required Python libraries and project modules.

In [ ]:
from pathlib import Path

import boto3
import pandas as pd
from sklearn.datasets import fetch_openml

from src.config import (
    BUCKET_NAME,
    RAW_DATA_KEY,
    AWS_REGION
)

## Download Dataset

The German Credit dataset is retrieved from OpenML and loaded into a pandas DataFrame.

In [ ]:
credit = fetch_openml(
    name="credit-g",
    version=1,
    as_frame=True
)

df = credit.frame

## Dataset Preview

Inspect the dataset structure and verify that the download was successful.

In [ ]:
df.head()

In [ ]:
df.info()

## Data Quality Validation

Perform a quick validation by checking the dataset shape and missing values.

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

missing_values = df.isnull().sum()

print(f"Total missing values: {missing_values.sum()}")

## Create Local Directories

Create the local directory used to store the raw dataset before uploading it to Amazon S3.

In [ ]:
Path("data/raw").mkdir(
    parents=True,
    exist_ok=True
)

## Upload to S3

Store a local copy of the validated dataset and upload it to the raw data layer in Amazon S3.

In [ ]:
local_file_name = "data/raw/german_credit.csv"

df.to_csv(
    local_file_name,
    index=False
)

In [ ]:
s3 = boto3.client("s3")

s3.upload_file(
    local_file_name,
    BUCKET_NAME,
    RAW_DATA_KEY
)

print(f"Uploaded to s3://{BUCKET_NAME}/{RAW_DATA_KEY}")

In [ ]:
response = s3.head_object(
    Bucket=BUCKET_NAME,
    Key=RAW_DATA_KEY
)

print("Upload successful")

## Result

The raw dataset is now stored in Amazon S3 and ready for feature engineering.